# M5 — Golden set, serving e feedback

Este notebook apresenta as evidências geradas por `src.evaluation.golden_set`. A API usa a política fixa aprovada por padrão porque o Thompson Sampling não passou no gate do M4.

In [1]:
import json
from pathlib import Path

import pandas as pd

# Localiza a raiz mesmo quando o kernel inicia dentro de notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / "configs").exists():
    project_root = project_root.parent
report_path = project_root / "reports/serving/golden_set_results.json"
assert report_path.exists(), "Execute python -m src.evaluation.golden_set antes."
report = json.loads(report_path.read_text(encoding="utf-8"))
report["readiness"]

{'status': 'ready',
 'active_policy_id': 'best_historical_action',
 'active_policy_version': '1.0.0',
 'policy_mode': 'approved_fixed_rollback',
 'model_version': '1.0.0',
 'm3_run_id': '7057c855256b4fffa1010cbc56708bfd',
 'm4_run_id': '19dccc266d674b36b8688c65e44acf11',
 'fallback_reason': 'Política adaptativa com status candidate; rollback fixo mantido.',
 'error': None}

## Cinco decisões revisadas

Os casos são sintéticos. A revisão humana explica limites e nunca transforma associações históricas em causalidade.

In [2]:
golden_table = pd.DataFrame(
    [
        {
            "case_id": case["case_id"],
            "scenario": case["scenario"],
            "action": case["recommended_action"],
            "fallback": case["used_fallback"],
            "review": case["human_review"]["status"],
            "contract_passed": case["contract_passed"],
        }
        for case in report["cases"]
    ]
)
golden_table

,case_id,scenario,action,fallback,review,contract_passed
0,golden_previous_success,Histórico anterior de sucesso,celular,False,faz_sentido,True
1,golden_previous_failure,Histórico anterior de fracasso,celular,False,faz_sentido,True
2,golden_never_contacted,Nenhum contato anterior,celular,False,requer_revisao,True
3,golden_unknown_audit_category,Categoria desconhecida somente na camada de au...,celular,False,faz_sentido,True
4,golden_cellular_unavailable,Melhor braço indisponível,telefone,True,requer_revisao,True


## Contrato operacional

- `POST /v1/recommendations` valida contexto, autorização e ações elegíveis.
- `POST /v1/feedback` aceita uma recompensa terminal por `recommendation_id`.
- `GET /health`, `/ready` e `/metrics` separam vida, prontidão e observabilidade.
- O SQLite guarda somente os dois campos de segmento necessários ao feedback, sem o payload completo.
- O modo `adaptive_demo` precisa ser solicitado explicitamente e não representa promoção.

In [3]:
# Mostra um payload reproduzível sem iniciar servidor ou gravar uma decisão.
example = json.loads(
    (project_root / "examples/recommendation_request.json").read_text(encoding="utf-8")
)
example

{'context': {'mes_contato': 'mai',
  'dia_semana': 'seg',
  'resultado_campanha_anterior': 'sucesso',
  'dias_desde_ultimo_contato': 3,
  'contatos_campanhas_anteriores': 2,
  'tentativas_anteriores_campanha_atual': 0,
  'taxa_variacao_emprego': -1.8,
  'indice_precos_consumidor': 92.893,
  'indice_confianca_consumidor': -46.2,
  'euribor_3_meses': 1.299,
  'numero_empregados': 5099.1,
  'nunca_contatado_anteriormente': 0},
 'eligible_actions': ['celular', 'telefone'],
 'contact_authorized': True,
 'do_not_contact': False}

## Conclusão

Os cinco contratos foram aprovados, incluindo fallback quando celular está indisponível. Isso comprova estabilidade técnica, não benefício causal ou autorização para contato real.